<a href="https://colab.research.google.com/github/JulianNguyen05/xiangqi-sorting-jetmax/blob/main/training/board_to_fen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BƯỚC 4 — ẢNH → THẾ CỜ (FEN): ĐÁNH GIÁ TRÊN TẬP TEST

**Đầu vào:** ảnh bàn cờ (tập test của `merged_v1`, chưa từng dùng khi train) + mô hình YOLO11n (Bước 2)
**Đầu ra:** thế cờ dạng FEN cho từng ảnh, và **độ chính xác của cả chuỗi ảnh → FEN** (không chỉ của riêng mô hình)
**Mã nguồn mới:** `vision/lattice.py` (khớp lưới 9 × 10 từ tâm quân), `board/draw.py` (vẽ bàn cờ)

### Vì sao cần notebook này?
Bước 2 đo mô hình bằng mAP — tính trên **từng quân**. Nhưng engine (Bước 5) cần **cả thế cờ đúng**: chỉ sai 1 trong 32 quân là FEN đã sai. Notebook này đo đúng thứ mà hệ thống dùng: *ảnh → FEN đúng bao nhiêu phần trăm*, sai kiểu gì, và luật cờ (Bước 5) cứu được bao nhiêu.

### Nội dung
1. Bài toán và phương pháp: vì sao khớp lưới từ **tâm quân** thay vì tìm bàn cờ
2. Đọc tập test, chọn ảnh có bàn cờ
3. Kiểm chứng thuật toán khớp lưới trên dữ liệu tổng hợp
4. Tạo **FEN thật** cho từng ảnh từ nhãn bbox
5. Chạy YOLO11n, ghép thành FEN dự đoán
6. So sánh FEN: 2 thiết lập (lưới từ nhãn thật / lưới từ chính dự đoán)
7. Luật cờ phát hiện được bao nhiêu FEN sai
8. Biết thế cờ lượt trước thì sửa được bao nhiêu (nối với Bước 5)
9. Chọn ngưỡng confidence
10. Ví dụ trực quan

---
# 1. Bài toán và phương pháp

Ảnh → FEN gồm 3 việc: (1) tìm **quân** (YOLO11n), (2) tìm **lưới 9 × 10 giao điểm** trong ảnh, (3) gán mỗi quân vào giao điểm gần nhất.

Việc (2) có hai cách:

| Cách | Ý tưởng | Ưu | Nhược | Dùng ở đâu |
|---|---|---|---|---|
| Tìm **bàn cờ** (`vision/board_detect.py`) | Lọc màu đường kẻ đỏ của tấm bàn nhựa → Hough → khớp mẫu lưới | Chính xác, thấy cả giao điểm trống | Chỉ chạy với **đúng loại bàn đã biết** (màu, kích thước) | Robot JetMax (bàn 36 × 26 cm của đề tài) |
| Khớp lưới từ **tâm quân** (`vision/lattice.py`, mới) | Quân luôn đứng trên giao điểm → tâm các quân nằm trên một lưới đều đã bị biến đổi phối cảnh → tìm phép biến đổi đó | **Không cần biết bàn cờ trông thế nào** | Cần ≥ 6–10 quân; thế cờ quá thưa có thể mơ hồ | Tập test (mỗi ảnh một kiểu bàn, góc chụp); dự phòng trên robot |

Tập test gồm ảnh từ 4 bộ dữ liệu khác nhau, mỗi bộ một kiểu bàn, màu đường kẻ, góc chụp — nên dùng cách thứ hai.

### Thuật toán khớp lưới (`vision/lattice.py`)

1. **Hướng trục và cỡ ô:** nối mỗi quân với 8 quân gần nhất; histogram góc của các vectơ cho 2 hướng trục chính. Độ dài các vectơ dọc một trục là **bội số nguyên** của cỡ ô $d$ → chọn $d$ lớn nhất sao cho mọi độ dài $\ell$ đều gần $k \cdot d$ ($k$ nguyên).
2. **Homography lưới → ảnh:** gán toạ độ nguyên cho các quân gần quân "hạt giống", khớp homography $H$, rồi **mở rộng dần bán kính** (2.5 → 14 ô) và khớp lại — chịu được phối cảnh (ô xa camera nhỏ hơn ô gần).
3. **Chọn lưới tốt nhất** trong các ứng viên: lưới giải thích được nhiều quân nhất; hoà thì chọn ô lớn nhất (lưới nửa ô cũng "khớp" mọi điểm).
4. **Chốt hướng và vị trí bàn bằng luật cờ:** thử mọi cách đặt (8 phép đối xứng × dịch chuyển), chỉ giữ cách đặt cùng chiều với ảnh thật (ảnh không bị lật gương), chọn cách cho thế cờ **ít vi phạm luật nhất** (tướng/sĩ trong cung, tượng không qua sông…), Đỏ ở hàng 0–4.
5. Quân nằm ngoài khung 9 × 10 hoặc lệch giao điểm > 0.35 ô → coi là **để ngoài bàn**.

In [ ]:
# ── Sơ đồ luồng xử lý (minh hoạ)
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
fig, ax = plt.subplots(figsize=(14, 2.6)); ax.axis('off'); ax.set_xlim(0, 14); ax.set_ylim(0, 2.6)
steps = [('Ảnh', '#e8e8e8'), ('YOLO11n\ntâm + loại quân', '#e8e8e8'), ('Khớp lưới 9×10\ntừ tâm quân', '#ffe0b2'),
         ('Chốt hướng bàn\nbằng luật cờ', '#ffe0b2'), ('Gán quân →\ngiao điểm', '#ffe0b2'), ('FEN', '#e8e8e8')]
for i, (t, c) in enumerate(steps):
    x = 0.2 + i * 2.3
    ax.add_patch(FancyBboxPatch((x, 0.7), 1.9, 1.2, boxstyle='round,pad=0.05', fc=c, ec='#555'))
    ax.text(x + 0.95, 1.3, t, ha='center', va='center', fontsize=10)
    if i < len(steps) - 1:
        ax.annotate('', xy=(x + 2.3, 1.3), xytext=(x + 1.95, 1.3), arrowprops=dict(arrowstyle='->', lw=1.5))
ax.text(7.0, 2.25, 'vision/lattice.py (tô cam)', ha='center', color='#e65100')
plt.show()

## Chuẩn bị môi trường

Cần trên Drive (giống các notebook trước): `MyDrive/datn/merged/merged_v1.zip` và `MyDrive/datn/runs/yolo11n_640_rot/weights/best.pt`. Kết quả ghi vào `MyDrive/datn/eval/`. Chạy được bằng CPU, nhưng nên bật GPU (Runtime → Change runtime type → T4) cho phần chạy YOLO.

In [ ]:
import os, sys, glob, time, random, shutil, subprocess
from collections import Counter
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
from PIL import Image

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not os.path.isdir('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    REPO = '/content/xiangqi-sorting-jetmax'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q', '--depth', '1',
                        'https://github.com/JulianNguyen05/xiangqi-sorting-jetmax', REPO], check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=False)
else:
    REPO = os.path.abspath('..') if os.path.exists('../vision/lattice.py') else os.path.abspath('.')
assert os.path.exists(f'{REPO}/vision/lattice.py'), 'Repo chưa có vision/lattice.py — hãy commit các file mới trước'
sys.path.insert(0, REPO)

from board import rules as R
from board.draw import draw_board
from board.geometry import NF, NR
from vision import lattice as L
from vision.pieces import CODES

BASE = '/content/drive/MyDrive/datn'
RUN_NAME = 'yolo11n_640_rot'                       # đổi nếu run của bạn có tên khác
PT = f'{BASE}/runs/{RUN_NAME}/weights/best.pt'
MERGED = '/content/merged'
if not os.path.isdir(f'{MERGED}/test/images'):
    print('Giải nén merged_v1.zip ...')
    shutil.unpack_archive(f'{BASE}/merged/merged_v1.zip', MERGED)
TEST_IMG, TEST_LBL = f'{MERGED}/test/images', f'{MERGED}/test/labels'
OUT = f'{BASE}/eval'
os.makedirs(OUT, exist_ok=True)
try:
    import ultralytics  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
print('Repo     :', REPO)
print('Mô hình  :', PT, '| có file:', os.path.exists(PT))
print('Tập test :', TEST_IMG, f'({len(os.listdir(TEST_IMG))} ảnh)')

---
# 2. Đọc tập test và chọn ảnh có bàn cờ

Mỗi ảnh có file nhãn YOLO (`lớp x_tâm y_tâm rộng cao`, toạ độ chuẩn hoá). Tên ảnh có dạng `<bộ>__<tên gốc>` (đặt ở Bước 1) nên biết được ảnh đến từ bộ dữ liệu nào.

Không phải ảnh nào cũng là bàn cờ: có ảnh chụp vài quân rời. Chỉ giữ ảnh có **≥ 10 quân** — đủ để khớp lưới và đủ giống một ván cờ thật.

In [ ]:
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
MIN_PIECES = 10

def read_labels(path, w, h):
    pts, codes, sizes = [], [], []
    for line in open(path):
        t = line.split()
        if len(t) < 5:
            continue
        c, xc, yc, bw, bh = int(t[0]), float(t[1]), float(t[2]), float(t[3]), float(t[4])
        pts.append((xc * w, yc * h)); codes.append(CODES[c]); sizes.append((bw * w + bh * h) / 2)
    return np.array(pts, float).reshape(-1, 2), codes, np.array(sizes)

data = {}
for p in sorted(glob.glob(f'{TEST_IMG}/*')):
    if not p.lower().endswith(IMG_EXT):
        continue
    stem = os.path.splitext(os.path.basename(p))[0]
    lp = f'{TEST_LBL}/{stem}.txt'
    if not os.path.exists(lp):
        continue
    w, h = Image.open(p).size
    pts, codes, sizes = read_labels(lp, w, h)
    data[stem] = dict(path=p, w=w, h=h, gt_pts=pts, gt_codes=codes, gt_size=sizes,
                      src=stem.split('__')[0] if '__' in stem else 'khác')

df = pd.DataFrame([{'nguồn': d['src'], 'số quân': len(d['gt_codes'])} for d in data.values()])
summary = df.groupby('nguồn')['số quân'].agg(['count', 'median', lambda s: (s >= MIN_PIECES).sum()])
summary.columns = ['số ảnh', 'số quân (trung vị)', f'ảnh có ≥ {MIN_PIECES} quân']
summary.loc['Tổng'] = [len(df), df['số quân'].median(), (df['số quân'] >= MIN_PIECES).sum()]
display(summary)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(df['số quân'], bins=range(0, 34), color='#4C72B0', edgecolor='white')
ax.axvline(MIN_PIECES - 0.5, color='#c62828', ls='--'); ax.text(MIN_PIECES, ax.get_ylim()[1] * 0.85, f' ngưỡng {MIN_PIECES} quân', color='#c62828')
ax.set_xlabel('số quân trong ảnh (theo nhãn)'); ax.set_ylabel('số ảnh'); ax.set_title('Phân bố số quân mỗi ảnh — tập test')
plt.tight_layout(); plt.show()

---
# 3. Kiểm chứng thuật toán khớp lưới trên dữ liệu tổng hợp

Trước khi dùng cho ảnh thật, kiểm tra `vision/lattice.py` trên dữ liệu **biết trước đáp án**: sinh thế cờ từ ván ngẫu nhiên → chiếu các giao điểm có quân lên ảnh với **góc xoay bất kỳ, phối cảnh ngẫu nhiên và nhiễu vị trí** (5% cỡ ô, giống sai số tâm bbox) → khớp lưới → so với thế cờ gốc.

Mức phối cảnh $p$: mỗi góc ảnh bị dịch ngẫu nhiên tới $p \times 25\%$ cạnh ảnh ($p = 0.1$: gần nhìn thẳng; $p = 0.3$: chụp xiên rõ rệt).

In [ ]:
RUN_SYNTH = True

def random_board(rnd):
    b, s = R.start_board(), R.RED
    for _ in range(rnd.randint(0, 80)):
        mv = R.legal_moves(b, s)
        if not mv:
            break
        caps = [m for m in mv if m[1] in b]
        m = rnd.choice(caps) if caps and rnd.random() < 0.4 else rnd.choice(mv)
        b, s = R.make_move(b, m), R.other(s)
    return b

def synth_points(b, rnd, persp, noise=0.05):
    cell, th = rnd.uniform(35, 65), rnd.uniform(0, 2 * np.pi)
    sq = np.array(list(b.keys()), float)
    P = np.c_[sq[:, 0] - 4, -(sq[:, 1] - 4.5)] * cell                     # nhìn từ trên xuống, Đỏ ở dưới
    P = P @ np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]]).T + 400
    j = lambda: rnd.uniform(-persp, persp) * 200
    src = np.float32([[0, 0], [800, 0], [800, 800], [0, 800]])
    dst = np.float32([[j(), j()], [800 + j(), j()], [800 + j(), 800 + j()], [j(), 800 + j()]])
    Hs = cv2.getPerspectiveTransform(src, dst)
    pts = cv2.perspectiveTransform(P.reshape(-1, 1, 2).astype(np.float32), Hs).reshape(-1, 2)
    return pts + np.random.normal(0, noise * cell, pts.shape), list(b.values())

if RUN_SYNTH:
    rnd = random.Random(0); np.random.seed(0)
    rows = []
    t0 = time.time()
    for persp in (0.1, 0.3):
        for _ in range(250):
            b = random_board(rnd)
            if len(b) < MIN_PIECES:
                continue
            pts, codes = synth_points(b, rnd, persp)
            got, fit, pl = L.image_to_board(pts, codes, [1.0] * len(codes))
            rows.append({'phối cảnh p': persp, 'nhóm': '10–19 quân' if len(b) < 20 else '≥ 20 quân', 'đúng': got == b})
    syn = pd.DataFrame(rows).pivot_table(index='nhóm', columns='phối cảnh p', values='đúng', aggfunc=['mean', 'count'])
    display(syn.style.format('{:.1%}', subset='mean'))
    print(f'({time.time() - t0:.0f} s; ~{(time.time() - t0) / len(rows) * 1000:.0f} ms mỗi thế cờ)')

> **Đọc kết quả:** với bàn **≥ 20 quân** (khai cuộc, trung cuộc) thuật toán khôi phục đúng **toàn bộ** thế cờ ở gần như mọi trường hợp, kể cả chụp xiên. Bàn **10–19 quân** khó hơn: khi các quân không đứng kề nhau, nhiều lưới khác nhau cùng giải thích được vị trí quân, luật cờ không phải lúc nào cũng phân xử được. Đây là giới hạn của cách khớp lưới từ quân (trên robot, `board_detect.py` thấy được cả giao điểm trống nên không gặp vấn đề này).

---
# 4. Tạo FEN thật cho từng ảnh test

Tập dữ liệu chỉ có nhãn **bbox** cho từng quân, không có FEN. Ta tạo **FEN thật** bằng cách chạy chính thuật toán khớp lưới lên **tâm các bbox nhãn** (vị trí và loại quân là của người gán nhãn), rồi chỉ giữ những ảnh mà kết quả **đáng tin**:

| Bước lọc | Lý do |
|---|---|
| Khớp được lưới | Ảnh có bàn cờ đủ rõ |
| ≥ 90% quân (theo nhãn) nằm đúng giao điểm | Lưới khớp tốt; cho phép vài quân để ngoài bàn |
| Thế cờ hợp lệ theo luật, cách đặt bàn không mơ hồ | Loại ảnh không phải ván cờ thật (quân bày tuỳ ý) hoặc lưới khớp sai |

⚠ Tập đánh giá vì vậy thiên về ảnh "dễ khớp lưới" hơn tập test gốc; ghi rõ điều này khi báo cáo.

In [ ]:
evalset, funnel = {}, Counter()
for k, d in data.items():
    funnel['1. Ảnh trong tập test'] += 1
    n = len(d['gt_codes'])
    if n < MIN_PIECES:
        continue
    funnel[f'2. Có ≥ {MIN_PIECES} quân'] += 1
    board, fit, pl = L.image_to_board(d['gt_pts'], d['gt_codes'], [1.0] * n)
    if board is None:
        continue
    funnel['3. Khớp được lưới'] += 1
    _, info = L.detections_to_board(d['gt_pts'], d['gt_codes'], [1.0] * n, fit, pl)
    if sum(i[0] is not None for i in info) < 0.9 * n:
        continue
    funnel['4. ≥ 90% quân nằm trên giao điểm'] += 1
    if pl.ambiguous or [i for i in R.validate(board) if i[0] != 'side_in_check']:
        continue
    funnel['5. Hợp lệ, không mơ hồ → TẬP ĐÁNH GIÁ'] += 1
    d.update(gt_board=board, fit=fit, pl=pl)
    evalset[k] = d

fdf = pd.DataFrame({'số ảnh': pd.Series(funnel)}).sort_index()
fdf['tỉ lệ'] = (fdf['số ảnh'] / fdf['số ảnh'].iloc[0]).map('{:.1%}'.format)
display(fdf)
src_count = pd.Series([d['src'] for d in evalset.values()]).value_counts().rename('ảnh trong tập đánh giá')
display(pd.concat([summary['số ảnh'].drop('Tổng'), src_count], axis=1).fillna(0).astype(int))
print('Số quân trung bình mỗi ảnh đánh giá: %.1f' % np.mean([len(d['gt_board']) for d in evalset.values()]))

Kiểm tra bằng mắt: lưới khớp được (chấm xanh = 90 giao điểm chiếu lên ảnh) và FEN thật tương ứng.

In [ ]:
def show_lattice(ax, d, fit=None, pl=None, title=''):
    img = cv2.cvtColor(cv2.imread(d['path']), cv2.COLOR_BGR2RGB)
    ax.imshow(img); ax.axis('off'); ax.set_title(title, fontsize=9)
    fit, pl = fit or d['fit'], pl or d['pl']
    grid = L.board_to_image(fit, pl, [(f, r) for r in range(NR) for f in range(NF)])
    ax.plot(grid[:, 0], grid[:, 1], '.', c='#00e676', ms=5)
    ax.set_xlim(0, img.shape[1]); ax.set_ylim(img.shape[0], 0)
    row0 = L.board_to_image(fit, pl, [(f, 0) for f in range(NF)])
    ax.plot(row0[:, 0], row0[:, 1], '-', c='#e53935', lw=2)

rnd = random.Random(1)
show = rnd.sample(list(evalset), min(3, len(evalset)))
fig, axes = plt.subplots(len(show), 2, figsize=(10, 4.6 * len(show)), squeeze=False)
for row, k in zip(axes, show):
    d = evalset[k]
    show_lattice(row[0], d, title=f"{d['src']} · {len(d['gt_codes'])} quân (đường đỏ = hàng 0, phía Đỏ)")
    draw_board(row[1], d['gt_board'], 'FEN thật:\n' + R.board_key(d['gt_board']))
plt.tight_layout(); plt.show()

---
# 5. Chạy YOLO11n và ghép thành FEN dự đoán

Chạy mô hình một lần cho mọi ảnh đánh giá với ngưỡng thấp (`conf ≥ 0.10`) và lưu lại; các ngưỡng cao hơn chỉ cần lọc lại (mục 9). Dùng `best.pt` (PyTorch) cho nhanh trên Colab — Bước 3 đã chứng minh `best.onnx` / `best.engine` cho cùng kết quả.

Hai thiết lập so sánh:

| Thiết lập | Lưới lấy từ | Đo cái gì |
|---|---|---|
| **A** | Nhãn thật (lưới "hoàn hảo") | Chỉ lỗi của **mô hình nhận diện quân** khi biết chính xác bàn cờ — tương ứng robot đã định vị được bàn (`board_detect.py`) |
| **B** | **Chính các quân mô hình dự đoán** (không dùng nhãn) | **Toàn bộ chuỗi ảnh → FEN**, kể cả lỗi khớp lưới và chốt hướng bàn |

In [ ]:
from ultralytics import YOLO
CONF_MIN, CONF = 0.10, 0.50                  # lưu mọi box >= 0.10; mặc định dùng 0.50 (giống notebook train)
model = YOLO(PT)
keys = list(evalset)
t0 = time.time()
for k, r in zip(keys, model.predict([evalset[k]['path'] for k in keys], conf=CONF_MIN, imgsz=640,
                                    verbose=False, stream=True)):
    b = r.boxes
    evalset[k]['pred_pts'] = b.xywh.cpu().numpy()[:, :2]
    evalset[k]['pred_cls'] = b.cls.cpu().numpy().astype(int)
    evalset[k]['pred_conf'] = b.conf.cpu().numpy()
print(f'{len(keys)} ảnh, {time.time() - t0:.1f} s')

def predict_board(d, conf, mode):
    m = d['pred_conf'] >= conf
    pts, codes, cf = d['pred_pts'][m], [CODES[c] for c in d['pred_cls'][m]], d['pred_conf'][m]
    if mode == 'A':
        return L.detections_to_board(pts, codes, cf, d['fit'], d['pl'])[0]
    return L.image_to_board(pts, codes, cf)[0]          # None nếu không khớp được lưới

def compare(gt, pr):
    """Các ô sai và loại lỗi giữa FEN thật và FEN dự đoán."""
    wrong, err = [], Counter()
    for sq in set(gt) | set(pr):
        g, p = gt.get(sq), pr.get(sq)
        if g == p:
            continue
        wrong.append(sq)
        err['thiếu quân' if p is None else 'thừa quân' if g is None else
            'sai màu' if g.isupper() != p.isupper() else 'sai loại'] += 1
    return wrong, err

for d in evalset.values():
    d['pred_A'] = predict_board(d, CONF, 'A')
    m = d['pred_conf'] >= CONF                                   # B: giữ cả lưới khớp được để vẽ ở mục 10
    d['pred_B'], d['fitB'], d['plB'] = L.image_to_board(d['pred_pts'][m], [CODES[c] for c in d['pred_cls'][m]], d['pred_conf'][m])
print('Xong ghép FEN cho 2 thiết lập.')

---
# 6. So sánh FEN dự đoán với FEN thật

| Chỉ số | Ý nghĩa |
|---|---|
| **Ảnh đúng hoàn toàn** | FEN dự đoán trùng 100% FEN thật — đây là điều engine cần |
| Sai ≤ 1 ô / ≤ 2 ô | Mức mà luật cờ có thể sửa khi biết thế trước (Bước 5, mục 5) |
| Độ chính xác theo quân | Tỉ lệ quân (theo nhãn) được nhận đúng ô, đúng loại, đúng màu |
| Thiếu / thừa / sai loại / sai màu | Kiểu lỗi, trung bình mỗi ảnh |
| Không khớp được lưới | Chỉ thiết lập B: quá ít quân được nhận diện hoặc lưới mơ hồ |

In [ ]:
def evaluate(mode, conf=CONF, keys=None):
    rows = []
    for k in keys or evalset:
        d = evalset[k]
        gt = d['gt_board']
        pr = d[f'pred_{mode}'] if conf == CONF else predict_board(d, conf, mode)
        if pr is None:
            rows.append(dict(ảnh=k, nguồn=d['src'], n=len(gt), khớp_lưới=False, sai=len(gt)))
            continue
        wrong, err = compare(gt, pr)
        rows.append(dict(ảnh=k, nguồn=d['src'], n=len(gt), khớp_lưới=True, sai=len(wrong), **err))
    t = pd.DataFrame(rows).fillna(0)
    for c in ['thiếu quân', 'thừa quân', 'sai loại', 'sai màu']:
        if c not in t:
            t[c] = 0
    return t

def summarize(t):
    ok = t[t['khớp_lưới']]
    per_piece = 1 - (ok['thiếu quân'] + ok['sai loại'] + ok['sai màu']).sum() / ok['n'].sum() if len(ok) else np.nan
    return pd.Series({
        'Ảnh đúng hoàn toàn': (t['sai'] == 0).mean(),
        'Sai ≤ 1 ô': (t['sai'] <= 1).mean(),
        'Sai ≤ 2 ô': (t['sai'] <= 2).mean(),
        'Độ chính xác theo quân': per_piece,
        'Thiếu quân / ảnh': ok['thiếu quân'].mean(),
        'Thừa quân / ảnh': ok['thừa quân'].mean(),
        'Sai loại / ảnh': ok['sai loại'].mean(),
        'Sai màu / ảnh': ok['sai màu'].mean(),
        'Không khớp được lưới': 1 - t['khớp_lưới'].mean(),
    })

tA, tB = evaluate('A'), evaluate('B')
res = pd.DataFrame({'A · lưới từ nhãn thật': summarize(tA), 'B · lưới từ dự đoán (end-to-end)': summarize(tB)})
pct = ['Ảnh đúng hoàn toàn', 'Sai ≤ 1 ô', 'Sai ≤ 2 ô', 'Độ chính xác theo quân', 'Không khớp được lưới']
display(res.style.format('{:.1%}', subset=pd.IndexSlice[pct, :]).format('{:.2f}', subset=pd.IndexSlice[[i for i in res.index if i not in pct], :]))
print(f'{len(tA)} ảnh đánh giá, ngưỡng conf = {CONF}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
bins = list(range(0, 8)) + [99]
for ax, (name, t) in zip(axes, [('A · lưới từ nhãn thật', tA), ('B · lưới từ dự đoán', tB)]):
    cnt = pd.cut(t['sai'], bins=[-0.5] + [b + 0.5 for b in bins], labels=[str(b) for b in bins[:-1]] + ['≥ 8']).value_counts(sort=False)
    ax.bar(cnt.index.astype(str), cnt.values / len(t) * 100, color=['#55A868'] + ['#DD8452'] * 2 + ['#C44E52'] * 6)
    ax.set_title(name); ax.set_xlabel('số ô sai trong ảnh'); ax.set_ylabel('% số ảnh')
    for x, v in zip(cnt.index.astype(str), cnt.values / len(t) * 100):
        if v > 0:
            ax.text(x, v + 0.5, f'{v:.0f}%', ha='center', fontsize=8)
plt.suptitle('Phân bố số ô sai mỗi ảnh (xanh: đúng hoàn toàn · cam: luật cờ còn sửa được · đỏ: quá nhiều lỗi)')
plt.tight_layout(); plt.show()

by_src = tB.groupby('nguồn').apply(lambda t: pd.Series({'số ảnh': len(t), 'đúng hoàn toàn': (t['sai'] == 0).mean(),
                                                        'sai ≤ 2 ô': (t['sai'] <= 2).mean()}))
display(by_src.style.format({'đúng hoàn toàn': '{:.1%}', 'sai ≤ 2 ô': '{:.1%}', 'số ảnh': '{:.0f}'}))

### Nhầm lẫn loại quân hay gặp nhất

Ở những ô cả nhãn và dự đoán đều có quân nhưng khác nhau (thiết lập A), quân nào hay bị nhận thành quân nào? Đối chiếu với lớp yếu nhất ở Bước 2 (`black_cannon`).

In [ ]:
NAME = {'k': 'tướng', 'a': 'sĩ', 'b': 'tượng', 'n': 'mã', 'r': 'xe', 'c': 'pháo', 'p': 'tốt'}
label = lambda c: f"{NAME[c.lower()]} {'Đỏ' if c.isupper() else 'Đen'}"
pairs = Counter()
for d in evalset.values():
    for sq, g in d['gt_board'].items():
        p = d['pred_A'].get(sq)
        if p is not None and p != g:
            pairs[(label(g), label(p))] += 1
conf_df = pd.DataFrame([{'nhãn thật': a, 'bị nhận thành': b, 'số lần': n} for (a, b), n in pairs.most_common(12)])
display(conf_df if len(conf_df) else 'Không có nhầm loại nào.')
miss = Counter(label(g) for d in evalset.values() for sq, g in d['gt_board'].items() if sq not in d['pred_A'])
display(pd.Series(miss, name='số lần bị bỏ sót').sort_values(ascending=False).head(8).to_frame())

---
# 7. Luật cờ phát hiện được bao nhiêu FEN sai?

`R.validate()` (Bước 5) kiểm tra thế cờ có thể xảy ra trong ván thật không. Không cần biết thế trước. Hai câu hỏi:
- Trong các FEN **sai**, bao nhiêu FEN bị luật cờ **phát hiện** (báo không hợp lệ → robot chụp lại)?
- Trong các FEN **đúng**, bao nhiêu bị **báo nhầm** (phải ≈ 0%, vì FEN thật đã được lọc hợp lệ ở mục 4)?

In [ ]:
rows = []
for name, mode in [('A', 'A'), ('B', 'B')]:
    flag_wrong, n_wrong, flag_ok, n_ok = 0, 0, 0, 0
    for d in evalset.values():
        pr = d[f'pred_{mode}']
        if pr is None:
            continue
        bad = bool([i for i in R.validate(pr) if i[0] != 'side_in_check'])
        if pr == d['gt_board']:
            n_ok += 1; flag_ok += bad
        else:
            n_wrong += 1; flag_wrong += bad
    rows.append({'thiết lập': name, 'FEN sai': n_wrong, 'luật cờ phát hiện': flag_wrong / max(n_wrong, 1),
                 'FEN đúng': n_ok, 'báo nhầm': flag_ok / max(n_ok, 1)})
display(pd.DataFrame(rows).style.format({'luật cờ phát hiện': '{:.1%}', 'báo nhầm': '{:.1%}'}))

> Luật cờ chỉ bắt được lỗi **làm thế cờ vô lý** (3 pháo, sĩ ra khỏi cung, tượng qua sông…). Lỗi "hợp lý" — bỏ sót 1 tốt, nhận mã thành xe ở ô mà xe cũng có thể đứng — vẫn lọt qua. Muốn bắt nốt cần **thế cờ lượt trước** (mục 8).

---
# 8. Biết thế cờ lượt trước: sửa được bao nhiêu? (nối với Bước 5)

Bước 5 đã đo bằng **lỗi mô phỏng**. Ở đây dùng **lỗi thật** của mô hình trên ảnh test:
1. Thế trước $S$ = FEN thật của ảnh (giả sử lượt trước đã xác nhận đúng).
2. Chọn ngẫu nhiên một nước hợp lệ $m$ của Đỏ → thế đúng sau nước đi $S' = S \oplus m$.
3. "Ảnh mới nhận diện được" = FEN dự đoán của **chính ảnh đó** (lỗi thật) với quân ở ô đi được chuyển sang ô đến.
4. `R.infer_move(S, Đỏ, ảnh mới)` → so với $m$.

So sánh với cách **tin thẳng ảnh** (dùng FEN dự đoán làm thế cờ mới).

In [ ]:
def infer_eval(mode, conf=CONF, seed=0, max_cost=2):
    rnd, out = random.Random(seed), Counter()
    for d in evalset.values():
        S = d['gt_board']
        moves = R.legal_moves(S, R.RED)
        if not moves:
            continue
        m = rnd.choice(moves)
        truth = R.make_move(S, m)
        pr = d[f'pred_{mode}'] if conf == CONF else predict_board(d, conf, mode)
        out['n'] += 1
        if pr is None:
            out['không chắc → chụp lại'] += 1
            continue
        obs = dict(pr)
        piece = obs.pop(m[0], None)
        obs.pop(m[1], None)
        if piece is not None:
            obs[m[1]] = piece
        out['tin thẳng ảnh: đúng'] += obs == truth
        r = R.infer_move(S, R.RED, obs)
        if r['cost'] > max_cost or r['ambiguous']:
            out['không chắc → chụp lại'] += 1
        elif r['move'] == m:
            out['suy ra: đúng nước'] += 1
        else:
            out['SAI mà không biết'] += 1
    n = out.pop('n')
    return pd.Series({k: out.get(k, 0) / n for k in ['tin thẳng ảnh: đúng', 'suy ra: đúng nước',
                                                       'không chắc → chụp lại', 'SAI mà không biết']}), n

iA, n = infer_eval('A'); iB, _ = infer_eval('B')
inf = pd.DataFrame({'A · lưới từ nhãn thật': iA, 'B · end-to-end': iB})
display(inf.style.format('{:.1%}')); print(f'{n} ảnh')

fig, ax = plt.subplots(figsize=(9, 3.6))
x = np.arange(len(inf)); w = 0.38
for i, (col, c) in enumerate(zip(inf.columns, ['#4C72B0', '#DD8452'])):
    ax.bar(x + (i - 0.5) * w, inf[col] * 100, w, label=col, color=c)
    for xi, v in zip(x, inf[col] * 100):
        ax.text(xi + (i - 0.5) * w, v + 1, f'{v:.0f}%', ha='center', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(inf.index); ax.set_ylabel('% số ảnh'); ax.set_ylim(0, 110); ax.legend()
ax.set_title('Ảnh thật, lỗi nhận diện thật: tin thẳng ảnh vs suy ra nước đi bằng luật cờ')
plt.tight_layout(); plt.show()

---
# 9. Chọn ngưỡng confidence

Ngưỡng thấp → ít bỏ sót nhưng nhiều báo nhầm; ngưỡng cao → ngược lại. Với bài toán FEN, chỉ số cần tối ưu là **tỉ lệ suy ra đúng nước** (mục 8, thiết lập B) — không phải mAP. Ô dưới mất vài phút (khớp lại lưới cho mỗi ngưỡng).

In [ ]:
THRS = [0.25, 0.35, 0.45, 0.55, 0.65, 0.75]
rows = []
t0 = time.time()
for thr in THRS:
    tAt, tBt = evaluate('A', thr), evaluate('B', thr)
    ib, _ = infer_eval('B', thr)
    rows.append({'ngưỡng': thr, 'A: đúng hoàn toàn': (tAt['sai'] == 0).mean(), 'B: đúng hoàn toàn': (tBt['sai'] == 0).mean(),
                 'B: suy ra đúng nước': ib['suy ra: đúng nước'], 'B: sai mà không biết': ib['SAI mà không biết']})
sweep = pd.DataFrame(rows).set_index('ngưỡng')
display(sweep.style.format('{:.1%}')); print(f'{time.time() - t0:.0f} s')
BEST_CONF = float(sweep['B: suy ra đúng nước'].idxmax())

fig, ax = plt.subplots(figsize=(8, 3.8))
for col, st in zip(sweep.columns, ['o--', 'o-', 's-', '^-']):
    ax.plot(sweep.index, sweep[col] * 100, st, label=col)
ax.axvline(BEST_CONF, color='#2e7d32', ls=':'); ax.text(BEST_CONF + 0.01, 5, f'chọn {BEST_CONF}', color='#2e7d32')
ax.set_xlabel('ngưỡng confidence'); ax.set_ylabel('% số ảnh'); ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=2)
plt.tight_layout(); plt.show()
print('Ngưỡng đề xuất cho robot:', BEST_CONF)

---
# 10. Ví dụ trực quan

Ảnh bên trái: lưới khớp từ chính các quân dự đoán (chấm vàng, thiết lập B) và các quân dự đoán (vòng xanh = vào đúng ô, đúng loại; đỏ = sai). Giữa: FEN thật. Phải: FEN dự đoán, ô sai tô đỏ. Ba ảnh: tốt nhất, trung vị, tệ nhất.

In [ ]:
def show_case(k, axes):
    d = evalset[k]
    pr = d['pred_B']
    img = cv2.cvtColor(cv2.imread(d['path']), cv2.COLOR_BGR2RGB)
    axes[0].imshow(img); axes[0].axis('off')
    m = d['pred_conf'] >= CONF
    if pr is not None:                                           # lưới B (từ chính dự đoán): chấm vàng
        g = L.board_to_image(d['fitB'], d['plB'], [(f, r) for r in range(NR) for f in range(NF)])
        axes[0].plot(g[:, 0], g[:, 1], '.', c='#ffd600', ms=4)
        _, info = L.detections_to_board(d['pred_pts'][m], [CODES[c] for c in d['pred_cls'][m]], d['pred_conf'][m], d['fitB'], d['plB'])
    else:
        info = [(None, 0)] * int(m.sum())
    for (x, y), c, (sq, _) in zip(d['pred_pts'][m], d['pred_cls'][m], info):
        good = sq is not None and d['gt_board'].get(sq) == CODES[c]
        axes[0].add_patch(plt.Circle((x, y), np.median(d['gt_size']) / 2, fill=False, lw=1.8, ec='#00c853' if good else '#ff1744'))
    axes[0].set_xlim(0, img.shape[1]); axes[0].set_ylim(img.shape[0], 0)
    wrong = compare(d['gt_board'], pr)[0] if pr is not None else []
    axes[0].set_title(f"{d['src']} · " + (f'sai {len(wrong)} ô' if pr is not None else 'không khớp được lưới'), fontsize=9)
    draw_board(axes[1], d['gt_board'], 'FEN thật')
    if pr is None:
        axes[2].axis('off'); axes[2].set_title('B: không khớp được lưới')
    else:
        draw_board(axes[2], pr, 'FEN dự đoán (B)', wrong=wrong)

tB_sorted = tB.sort_values('sai')
picks = [tB_sorted['ảnh'].iloc[0], tB_sorted['ảnh'].iloc[len(tB_sorted) // 2], tB_sorted['ảnh'].iloc[-1]]
fig, axes = plt.subplots(3, 3, figsize=(14, 15))
for row, k in zip(axes, picks):
    show_case(k, row)
plt.tight_layout(); plt.show()

---
# Lưu kết quả và tổng kết số liệu

In [ ]:
tB.merge(tA[['ảnh', 'sai']].rename(columns={'sai': 'sai_A'}), on='ảnh').to_csv(f'{OUT}/board_to_fen_per_image.csv', index=False)
final = pd.Series({
    'Ảnh trong tập test': funnel['1. Ảnh trong tập test'],
    'Ảnh trong tập đánh giá (có FEN thật)': len(evalset),
    'Ngưỡng conf dùng ở mục 6–8': CONF,
    'A: ảnh → FEN đúng hoàn toàn': f"{(tA['sai'] == 0).mean():.1%}",
    'B: ảnh → FEN đúng hoàn toàn (end-to-end)': f"{(tB['sai'] == 0).mean():.1%}",
    'B: độ chính xác theo quân': f"{summarize(tB)['Độ chính xác theo quân']:.1%}",
    'B: tin thẳng ảnh → đúng thế cờ sau nước đi': f"{iB['tin thẳng ảnh: đúng']:.1%}",
    'B: suy ra bằng luật → đúng nước': f"{iB['suy ra: đúng nước']:.1%}",
    'B: sai mà không biết': f"{iB['SAI mà không biết']:.1%}",
    'Ngưỡng conf đề xuất (mục 9)': BEST_CONF,
}, name='giá trị')
display(final.to_frame())
print('Đã lưu', f'{OUT}/board_to_fen_per_image.csv')

### Nhận xét (đọc cùng bảng trên)

- **Mô hình tốt theo mAP chưa chắc cho FEN đúng:** mAP@0.5 = 0.968 ở Bước 2 là trên từng quân; với 20–32 quân mỗi ảnh, xác suất *cả bàn* đúng thấp hơn nhiều. Chênh lệch giữa "độ chính xác theo quân" và "ảnh đúng hoàn toàn" thể hiện điều này.
- **A so với B** cho biết lỗi đến từ đâu: A thấp → do mô hình nhận diện quân; B thấp hơn A nhiều → do khớp lưới (thường ở ảnh ít quân hoặc mô hình bỏ sót nhiều quân).
- **Luật cờ + thế trước** (mục 8) là phần nâng tỉ lệ đúng lên mức dùng được cho robot; quan trọng hơn, tỉ lệ **sai mà không biết** phải gần 0 — hệ thống thà yêu cầu chụp lại còn hơn đi theo thế cờ sai.

### Hạn chế
1. Ảnh test là ảnh công khai (nhiều kiểu bàn, góc chụp), **không phải camera robot**; con số trên robot có thể khác (thường tốt hơn vì bàn và ánh sáng cố định).
2. "FEN thật" suy từ nhãn bbox + khớp lưới; ảnh khớp lưới kém đã bị loại (mục 4) nên tập đánh giá thiên về ảnh dễ.
3. Khớp lưới từ quân giả sử ô vuông đều (sông rộng 1 ô); với thế cờ thưa (< 15 quân) có thể mơ hồ — trên robot dùng `board_detect.py` (thấy cả giao điểm trống) làm chính, `lattice.py` làm dự phòng.
4. Nước đi ở mục 8 là mô phỏng (chọn ngẫu nhiên), lỗi nhận diện là thật.

---
# Tài liệu tham khảo (Sources)

1. Hartley R., Zisserman A. — *Multiple View Geometry in Computer Vision* (2nd ed.), Cambridge University Press, 2004 — homography, biến đổi phối cảnh.
2. OpenCV — `findHomography`, `perspectiveTransform`: https://docs.opencv.org/4.x/d9/d0c/group__calib3d.html
3. Ultralytics — *Predict mode*: https://docs.ultralytics.com/modes/predict/
4. Fischler M. A., Bolles R. C. — *Random Sample Consensus*, Communications of the ACM 24(6), 1981 (ý tưởng chọn mô hình giải thích được nhiều điểm nhất).
5. Các notebook của đề tài: `build_dataset.ipynb` (tập test, tên ảnh), `train_yolo11.ipynb` (mô hình), `rules_engine.ipynb` (luật cờ, `infer_move`).